In [ ]:
ARAZI_KATEGORILERI = {
    "Kategori 0": {
        "Arazi kategorisi": "Açık deniz etkisine maruz deniz veya kıyı alanı",
        "z0": 0.003,
        "zmin": 1.0
    },
    "Kategori I": {
        "Arazi kategorisi": "Göl, düz ve açık arazi: Göller veya ihmal edilebilecek seviyede bitki örtüsü olan ve engebeli olmayan düz ve yatay alan",
        "z0": 0.01,
        "zmin": 1.0
    },
    "Kategori II": {
        "Arazi kategorisi": "Bitki örtüsü ve/veya binalarla kaplı arazi: Çayır gibi az seviyede bitki örtüsü olan ve aralarında en az engel yüksekliğinin 20 katı kadar mesafe bulunan engellere (ağaçlar, binalar) sahip alan",
        "z0": 0.05,
        "zmin": 2.0
    },
    "Kategori III": {
        "Arazi kategorisi": "Orman, sanayi veya kentsel bölge: Düzgün yayılı şekilde bir bitki örtüsüne veya binalara veya aralarında en az engel yüksekliğinin 20 katı kadar mesafe bulunan engellere sahip alan (kasabalar, yörekent, ormanlık alan gibi",
        "z0": 0.3,
        "zmin": 5.0
    },
    "Kategori IV": {
        "Arazi kategorisi": "Yoğun şehir merkezi: Yüzeyinin en az % 15’i, yükseklik ortalaması 15 m’yi aşan binalarla kaplı alan",
        "z0": 1.0,
        "zmin": 10.0
    }
}

# edge.py
import math
from dataclasses import dataclass, field
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Union
from enum import Enum

EPS = 1.0e-9


class SurfaceType(Enum):
    WALL = "WALL"
    ROOF = "ROOF"
    nodata = ""


class WindRelation(Enum):
    WINDWARD = "WINDWARD"
    LEEWARD = "LEEWARD"
    PARALLEL = "PARALLEL"
    nodata = ""


@dataclass
class Zone:
    name: str
    polygon: np.ndarray
    cpe_neg: Optional[float] = None
    cpe_pos: Optional[float] = None


def polygon_centroid(polygon: np.ndarray) -> np.ndarray:
    """Poligonun merkezini hesaplar"""
    pts = np.asarray(polygon, dtype=float)

    if np.allclose(pts[0], pts[-1]):
        pts = pts[:-1]

    return np.mean(pts, axis=0)
    

@dataclass
class Edge:
    """
    Saf geometrik kenar verisi. Rüzgar analizi EdgeAnalyzer tarafından yapılır.
    """
    index: int
    p1: np.ndarray
    p2: np.ndarray

    vector: np.ndarray
    length: float
    unit_vector: np.ndarray

    vector_xy: np.ndarray
    length_xy: float
    direction_xy: Optional[np.ndarray]

    # EdgeAnalyzer tarafından doldurulur
    pos_front: Optional[float] = None
    pos_back: Optional[float] = None
    angle: Optional[float] = None
    exposed: bool = False
    leading: bool = False
    vertical: bool = False

    shared: List[Dict[str, Any]] = field(default_factory=list)
    same_axis: List[Dict[str, Any]] = field(default_factory=list)

    @classmethod
    def from_points(cls, index: int, p1, p2) -> "Edge":
        p1 = np.asarray(p1, dtype=float).copy()
        p2 = np.asarray(p2, dtype=float).copy()

        vector = p2 - p1
        length = float(np.linalg.norm(vector))
        if length < 1e-12:
            raise ValueError(f"Çökmüş kenar: {index}")

        unit_vector = vector / length

        vector_xy = vector[:2].copy()
        length_xy = float(np.linalg.norm(vector_xy))

        direction_xy = None if length_xy < 1e-12 else vector_xy / length_xy

        return cls(
            index=index,
            p1=p1, p2=p2,
            vector=vector,
            length=length,
            unit_vector=unit_vector,
            vector_xy=vector_xy,
            length_xy=length_xy,
            direction_xy=direction_xy,
        )

    # ------------------------------------------------------------------
    # Geometrik karşılaştırma (rüzgardan bağımsız)
    # ------------------------------------------------------------------

    def same_axis_xy(
        self,
        other: "Edge",
        angle_tol: float = 2.0,
        distance_tol: float = 1e-6,
    ) -> bool:
        d1, d2 = self.direction_xy, other.direction_xy
        if d1 is None or d2 is None:
            return False

        dot = float(np.clip(abs(np.dot(d1, d2)), -1.0, 1.0))
        if np.degrees(np.arccos(dot)) > angle_tol:
            return False

        normal = np.array([-d1[1], d1[0]])
        distance = abs(float(np.dot(other.p1[:2] - self.p1[:2], normal)))
        return distance <= distance_tol

    def is_same_edge(self, other: "Edge", tol: float = 1e-6) -> bool:
        return (
            np.allclose(self.p1, other.p1, atol=tol)
            and np.allclose(self.p2, other.p2, atol=tol)
        ) or (
            np.allclose(self.p1, other.p2, atol=tol)
            and np.allclose(self.p2, other.p1, atol=tol)
        )

    def reset_analysis(self) -> None:
        """Rüzgar analizine bağlı alanları sıfırla."""
        self.pos_front = None
        self.pos_back = None
        self.angle = None
        self.exposed = False
        self.leading = False
        self.vertical = (self.direction_xy is None)
        self.shared = []
        self.same_axis = []

# edge_analyzer.py



class EdgeAnalyzer:
    """
    Bir yüzeyin Edge koleksiyonunu, verilen rüzgâr yönü için
    XY izdüşümünde analiz eder.

    Konvansiyon:
        wind_vector = rüzgârın hareket yönü

    Örneğin:
        [0, 1, 0] -> rüzgâr +Y yönünde hareket eder.
        Bu durumda rüzgâra ilk bakan taraf -Y tarafıdır.

    Analizler:
        - edge 2D metrikleri
        - exposed
        - leading
        - diğer çatılarla same_axis
        - shared edge
    """

    def __init__(
        self,
        edges: Dict[int, "Edge"],
        surface_normal: np.ndarray,
        is_ccw: bool,
        wind_relation: str = "",
        tol: float = 1e-6,
    ):
        self.edges = edges
        self.surface_normal = np.asarray(surface_normal, dtype=float)
        self.is_ccw = bool(is_ccw)
        self.wind_relation = wind_relation
        self.tol = float(tol)

        self._wind_to_2d: Optional[np.ndarray] = None

    # ------------------------------------------------------------------
    # PUBLIC
    # ------------------------------------------------------------------

    def analyze(
        self,
        wind_vector: np.ndarray,
        building: Any = None,
        current_surface_name: str = "",
    ) -> Dict[int, "Edge"]:
        """
        Kenarları verilen rüzgâr yönüne göre analiz eder.

        wind_vector:
            Rüzgârın hareket yönü.
            Örneğin [0, 1, 0] -> +Y.
        """

        w_to = self._normalize_wind(wind_vector)

        if w_to is None:
            return {}

        self._wind_to_2d = w_to

        # --------------------------------------------------------------
        # 1) Kenarları sıfırla ve metrikleri hesapla
        # --------------------------------------------------------------

        for edge in self.edges.values():
            edge.reset_analysis()
            self._compute_edge_metrics(edge)

        # --------------------------------------------------------------
        # 2) Leading kenarları belirle
        # --------------------------------------------------------------

        self._mark_leading()

        # --------------------------------------------------------------
        # 3) Diğer çatılarla karşılaştır
        # --------------------------------------------------------------

        if building is not None:
            self._find_shared_and_same_axis(
                building,
                current_surface_name,
            )

        return self.edges

    # ------------------------------------------------------------------
    # 1) KENAR METRİKLERİ
    # ------------------------------------------------------------------

    def _compute_edge_metrics(self, edge: "Edge") -> None:
        """
        Kenarın XY izdüşümüne ait metrikleri hesaplar.
        """

        # --------------------------------------------------------------
        # XY'de uzunluğu olmayan kenar
        # --------------------------------------------------------------

        if edge.direction_xy is None:
            pos = float(
                np.dot(
                    edge.p1[:2],
                    self._wind_to_2d,
                )
            )

            edge.pos_front = pos
            edge.pos_back = pos
            edge.angle = 90.0
            edge.exposed = False
            edge.vertical = True

            return

        # --------------------------------------------------------------
        # Rüzgâr doğrultusundaki konum
        #
        # wind_to yönünde ilerledikçe projeksiyon büyür.
        #
        # Dolayısıyla:
        #
        #     küçük projection -> rüzgâra daha yakın / front
        #     büyük projection  -> daha geride / back
        # --------------------------------------------------------------

        s1 = float(
            np.dot(
                edge.p1[:2],
                self._wind_to_2d,
            )
        )

        s2 = float(
            np.dot(
                edge.p2[:2],
                self._wind_to_2d,
            )
        )

        edge.pos_front, edge.pos_back = (
            (s1, s2)
            if s1 <= s2
            else (s2, s1)
        )

        # --------------------------------------------------------------
        # Kenar ile rüzgâr arasındaki açı
        #
        # 0°  -> paralel
        # 90° -> dik
        # --------------------------------------------------------------

        cos_a = float(
            np.clip(
                abs(
                    np.dot(
                        edge.direction_xy,
                        self._wind_to_2d,
                    )
                ),
                -1.0,
                1.0,
            )
        )

        edge.angle = float(
            np.degrees(
                np.arccos(cos_a)
            )
        )

        # --------------------------------------------------------------
        # Exposed
        # --------------------------------------------------------------

        edge.exposed = self._is_exposed(edge)
        edge.vertical = False

    # ------------------------------------------------------------------
    # 2) EXPOSED
    # ------------------------------------------------------------------

    def _is_exposed(self, edge: "Edge") -> bool:
        """
        Kenarın rüzgâra bakan dış kenar olup olmadığını belirler.

        wind_to:
            Rüzgârın hareket yönüdür.

        CCW poligonda dış normal kenarın sağındadır.
        CW poligonda dış normal kenarın solundadır.

        Cross product işareti:
            CCW -> cross > 0  => exposed
            CW  -> cross < 0  => exposed
        """

        e = edge.p2[:2] - edge.p1[:2]
        w = self._wind_to_2d

        cross = (
            e[0] * w[1]
            - e[1] * w[0]
        )

        if self.is_ccw:
            return cross > self.tol

        return cross < -self.tol

    # ------------------------------------------------------------------
    # 3) LEADING
    # ------------------------------------------------------------------

    def _mark_leading(self) -> None:
        """
        Exposed kenarlar arasından rüzgâr yönünde en önde bulunan
        kenar(lar)ı leading olarak işaretler.

        wind_to yönünde küçük projeksiyon daha öndedir.
        """

        exposed = [
            edge
            for edge in self.edges.values()
            if edge.exposed
        ]

        if not exposed:
            return

        global_front = min(
            edge.pos_front
            for edge in exposed
        )

        for edge in exposed:
            if (
                abs(edge.pos_front - global_front)
                <= self.tol
            ):
                if edge.length_xy > self.tol:
                    edge.leading = True

    # ------------------------------------------------------------------
    # 4) SHARED / SAME_AXIS
    # ------------------------------------------------------------------

    def _find_shared_and_same_axis(
        self,
        building: Any,
        current_surface_name: str,
    ) -> None:
        """
        Diğer çatı yüzeyleriyle aynı eksen üzerindeki ve ortak olan
        kenarları belirler.
        """

        surfaces_items = getattr(
            building,
            "surfaces_items",
            {},
        ) or {}

        other_roofs: List[Tuple[str, Any]] = []

        for name, surface in surfaces_items.items():

            if name == current_surface_name:
                continue

            surface_type = getattr(
                surface.surface_type,
                "value",
                surface.surface_type,
            )

            if surface_type != "ROOF":
                continue

            other_roofs.append(
                (name, surface)
            )

        if not other_roofs:
            return

        # --------------------------------------------------------------
        # Her kenarı diğer çatılarla karşılaştır
        # --------------------------------------------------------------

        for edge in self.edges.values():

            if edge.direction_xy is None:
                continue

            for other_name, other_surface in other_roofs:

                for other_edge in other_surface.edges.values():

                    # Aynı fiziksel XY ekseninde değillerse geç
                    if not edge.same_axis_xy(other_edge):
                        continue

                    is_same = edge.is_same_edge(
                        other_edge,
                        tol=self.tol,
                    )

                    entry = {
                        "surface": other_name,
                        "surface_edge": other_edge.index,
                        "surface_angle": getattr(
                            other_surface,
                            "angle",
                            None,
                        ),
                        "direction_xy": other_edge.direction_xy,
                        "same_edge": is_same,
                    }

                    edge.same_axis.append(entry)

                    if is_same:
                        edge.shared.append(entry)

    # ------------------------------------------------------------------
    # YARDIMCI
    # ------------------------------------------------------------------

    @staticmethod
    def _normalize_wind(
        wind_vector: np.ndarray,
    ) -> Optional[np.ndarray]:
        """
        3D rüzgâr vektörünün XY izdüşümünü normalize eder.
        """

        w = np.asarray(
            wind_vector,
            dtype=float,
        )

        if w.size < 2:
            return None

        w2 = w[:2]

        norm = np.linalg.norm(w2)

        if norm < 1e-12:
            return None

        return w2 / norm

# windplane.py


class WindPlane:
    def __init__(self, polygon: Union[List, np.ndarray], name: str = "Test"):
        """3B Düzlemsel Poligon Nesnesi"""
        pts = np.asarray(polygon, dtype=float)

        if len(pts) > 1 and np.allclose(pts[0], pts[-1], atol=1e-8):
            pts = pts[:-1]

        if len(pts) < 3:
            raise ValueError("Poligon en az 3 noktadan oluşmalıdır.")

        self.polygon_name = name
        self.coords = pts
        self.pts_3d: np.ndarray = self.close_polygon(pts)
        self.n_pts: int = len(pts)

        self.normal, self.normal_unit = self._compute_normal()
        self.centroid = polygon_centroid(self.pts_3d)
        self.pitch: float = self._compute_pitch()
        self.surface_type: SurfaceType = SurfaceType.ROOF if self.pitch <= 75.0 else SurfaceType.WALL
        self.proj_info = self._get_best_projection_plane()
        self.angle = self._compute_angle()
        self.exposed_edge_list = []
        self.edges = self._build_edges()
        self.global_leading = False

        self.properties = {
            "name": self.polygon_name,
            "polygon": self.pts_3d,
            "surface_type": self.surface_type,
            "angle": self.angle,
            "pitch": self.pitch,
            "coords": pts,
            "zmin": float(np.min(pts[:, 2])),
            "zmax": float(np.max(pts[:, 2])),
            "wind_vector": None,
            "wind_relation": WindRelation.nodata,
            "global_leading": None
        }

    @staticmethod
    def close_polygon(polygon: np.ndarray, tol: float = 1e-8) -> np.ndarray:
        """Poligonu kapatır"""
        polygon = np.asarray(polygon, dtype=float)
        if len(polygon) == 0:
            return polygon
        if not np.allclose(polygon[0], polygon[-1], atol=tol):
            polygon = np.vstack([polygon, polygon[0]])
        return polygon

    def _compute_normal(self) -> Tuple[np.ndarray, np.ndarray]:
        normal = np.zeros(3)
        for i in range(self.n_pts):
            p_curr, p_next = self.pts_3d[i], self.pts_3d[(i + 1) % self.n_pts]
            normal[0] += (p_curr[1] - p_next[1]) * (p_curr[2] + p_next[2])
            normal[1] += (p_curr[2] - p_next[2]) * (p_curr[0] + p_next[0])
            normal[2] += (p_curr[0] - p_next[0]) * (p_curr[1] + p_next[1])

        norm_len = np.linalg.norm(normal)
        if norm_len < 1e-12:
            raise ValueError(f"Geçersiz/Çökmüş poligon düzlemi: {self.pts_3d}")
        return normal, normal / norm_len

    def _compute_angle(self) -> float:
        if self.pitch < 1e-10:
            return 0.0
        nx, ny, nz = self.normal_unit
        if self.pitch >= 90.0 - 1e-10:
            return float(np.degrees(np.arctan2(ny, nx)))
        horiz_len = np.hypot(nx, ny)
        if horiz_len < 1e-12:
            return 0.0
        sign = np.sign(ny) if abs(ny) > abs(nx) else np.sign(nx)     
        return float(sign * self.pitch)

    def _compute_pitch(self) -> float:
        nz_ratio = abs(self.normal_unit[2])
        nz_ratio = np.clip(nz_ratio, -1.0, 1.0)
        return float(np.degrees(np.arccos(nz_ratio)))

    def _get_best_projection_plane(self):
        abs_n = np.abs(self.normal_unit)
        max_idx = np.argmax(abs_n)
        if max_idx == 2:
            return 'XY', (0, 1), 2
        elif max_idx == 1:
            return 'XZ', (0, 2), 1
        else:
            return 'YZ', (1, 2), 0

    def analyze_wind_relation(self, w_vector: np.ndarray) -> WindRelation:
        w_xy = np.asarray(w_vector[:2], dtype=float)
        w_norm = np.linalg.norm(w_xy)
        if w_norm < 1e-12:
            raise ValueError("Rüzgar vektörünün yatay bileşeni sıfır olamaz.")

        w_xy /= w_norm
        n_xy = self.normal_unit[:2]
        dot = np.dot(n_xy, w_xy)

        if dot < -0.2:
            return WindRelation.WINDWARD
        elif dot > 0.2:
            return WindRelation.LEEWARD
        else:
            return WindRelation.PARALLEL

    def _build_edges(self) -> dict:
        edges = {}

        for i in range(self.n_pts):
            p1 = self.pts_3d[i]
            p2 = self.pts_3d[(i + 1) % self.n_pts]

            vector = p2 - p1
            length = np.linalg.norm(vector)

            if length < 1e-12:
                continue

            unit_vector = vector / length

            # XY izdüşümü
            vector_xy = vector[:2]
            length_xy = np.linalg.norm(vector_xy)

            if length_xy < 1e-12:
                direction_xy = None
            else:
                direction_xy = vector_xy / length_xy

            edges[i] = {
                "p1": p1.copy(),
                "p2": p2.copy(),

                # 3B geometri
                "vector": vector.copy(),
                "length": float(length),
                "unit_vector": unit_vector.copy(),

                # XY izdüşümü
                "vector_xy": vector_xy.copy(),
                "length_xy": float(length_xy),
                "direction_xy": (
                    direction_xy.copy()
                    if direction_xy is not None
                    else None
                ),
            }

        return edges
    def _unproject_to_3d(self, pts_2d: np.ndarray) -> np.ndarray:
        _, (idx1, idx2), missing_idx = self.proj_info
        n_missing = self.normal_unit[missing_idx]
        D = -np.dot(self.normal_unit, self.pts_3d[0])

        pts_3d = []
        for pt in pts_2d:
            p3d = np.zeros(3)
            p3d[idx1] = pt[0]
            p3d[idx2] = pt[1]
            known_sum = self.normal_unit[idx1] * pt[0] + self.normal_unit[idx2] * pt[1] + D
            p3d[missing_idx] = -known_sum / n_missing
            pts_3d.append(p3d)
        return np.array(pts_3d)

    def polygon_direction_xy(self) -> str:
        """Poligonun XY düzlemindeki yönünü belirler."""
        polygon = self.pts_3d

        if np.allclose(polygon[0], polygon[-1]):
            polygon = polygon[:-1]

        area2 = 0.0
        n = len(polygon)

        for i in range(n):
            x1, y1 = polygon[i][:2]
            x2, y2 = polygon[(i + 1) % n][:2]
            area2 += x1 * y2 - x2 * y1

        if area2 > EPS:
            return "CCW"
        elif area2 < -EPS:
            return "CW"
        else:
            return "DEGENERATE"

    def _build_edges(self) -> Dict[int, Edge]:
        edges = {}
        for i in range(self.n_pts):
            p1 = self.pts_3d[i]
            p2 = self.pts_3d[(i + 1) % self.n_pts]
            try:
                edges[i] = Edge.from_points(i, p1, p2)
            except ValueError:
                continue
        return edges

    def _analyze_edges(self, w, building=None) -> Dict[int, Edge]:
        is_ccw = (self.polygon_direction_xy() == "CCW")
        rel = self.analyze_wind_relation(w)

        analyzer = EdgeAnalyzer(
            edges=self.edges,
            surface_normal=self.normal_unit,
            is_ccw=is_ccw,
            wind_relation=rel.value,   # "WINDWARD" / "LEEWARD" / "PARALLEL"
            tol=1e-6,
        )
        return analyzer.analyze(
            wind_vector=w,
            building=building,
            current_surface_name=self.polygon_name,
        )

    def analysis_(self, w, building) -> None:
        rel_ = self.analyze_wind_relation(w)
        self.wind_relation = rel_

        self.edges = self._analyze_edges(w, building)

        self.properties["polygon_direction_xy"] = self.polygon_direction_xy()
        self.properties["wind_relation"] = rel_
        self.properties["wind_vector"] = w
        self.properties["any_shared"] = any(e.shared for e in self.edges.values())
        
class BuildingWindEngine:
    

    def __init__(
        self, 
        points: Dict[str, Tuple[float, float, float]], 
        polygons: Dict[str, List[str]], 
        v_b0: float = 28.0, 
        terrain: str = "Kategori III",
        w_dir: List[float] = [1.0, 0.0, 0.0],
        scale_factor: float = 1000.0,
        rho: float = 1.25,
        ct: float = 1.0,
        kI: float = 1.0
    ):
        self.raw_points = points
        self.polygons = polygons
        self.v_b0 = float(v_b0)
        self.terrain = terrain
        self.w_dir = np.array(w_dir, dtype=float)
        self.scale_factor = float(scale_factor)
        self.rho = rho
        self.ct = ct
        self.kI = kI
        self.e= None
        self.all_roof_polygons = {}

        if self.terrain not in ARAZI_KATEGORILERI:
            raise ValueError(f"Geçersiz arazi kategorisi: {self.terrain}")
        
        self.z0 = ARAZI_KATEGORILERI[self.terrain]["z0"]
        self.zmin = ARAZI_KATEGORILERI[self.terrain]["zmin"]

        # 1. Noktaları ölçekle
        self.scaled_points = self._scale_points()
        
        # 2. Yüzeyleri analiz et (Duvar/Çatı ayrımı ve kotlar)
        self.surfaces = self._analyze_surfaces()
        
        # 3. Yüzeylerden gelen verilerle bina geometrisini bağla (Artık çakışma yok)
        self.geometry = self._calculate_building_geometry()
        
        # 4. Rüzgar parametrelerini (q_b, q_p) hesapla
        self._calculate_wind_parameters()

    def _scale_points(self) -> Dict[str, np.ndarray]:
        return {
            name: np.array(pt, dtype=float) / self.scale_factor 
            for name, pt in self.raw_points.items()
        }

    def _analyze_surfaces(self, w= None) -> Dict[str, Dict[str, Any]]:
        """Poligon bağımsız yüzey tipini ve kotlarını çıkarır."""
        if w is None:
            w = self.w_dir
            
        surfaces = {}
        self.surfaces_items= {}
        self.all_roof_polygons = {}
        for poly_name, pt_names in self.polygons.items():
            pts = np.array([self.scaled_points[pt] for pt in pt_names])
            surface_= WindPlane(pts, name=poly_name)
            self.surfaces_items[poly_name] = surface_
            if surface_.surface_type.value=="ROOF":
                self.all_roof_polygons[poly_name]= surface_.properties['coords']


            surfaces[poly_name] = {
                "type": surface_.properties['surface_type'],
                "pitch": surface_.properties['pitch'],
                "relation": surface_.analyze_wind_relation(w),
                "zmin": surface_.properties['zmin'],
                "zmax": surface_.properties['zmax'],
                "coords": surface_.properties['coords']
            }
        return surfaces

    def _calculate_building_geometry(self) -> Dict[str, float]:
        """Hazır self.surfaces üzerinden saçak kotunu çekerek geometriyi tamamlar."""
        pts_matrix = np.array(list(self.scaled_points.values()))

        # Rüzgar eksenleri
        w_xy = self.w_dir[:2] / np.linalg.norm(self.w_dir[:2])
        v_vec = np.array([-w_xy[1], w_xy[0]])

        u_vals = [np.dot(pt[:2], w_xy) for pt in pts_matrix]
        v_vals = [np.dot(pt[:2], v_vec) for pt in pts_matrix]

        b = max(v_vals) - min(v_vals)
        d = max(u_vals) - min(u_vals)
        
        z_ground = float(min(pt[2] for pt in pts_matrix))
        z_ridge = float(max(pt[2] for pt in pts_matrix))

        # Saçak kotu tespiti (Duvarların maksimum z kotu)
        wall_z_maxs = [s["zmin"] for s in self.surfaces.values() if s["type"] == SurfaceType.ROOF]
        z_eaves = max(wall_z_maxs) if wall_z_maxs else z_ridge

        h_total = z_ridge - z_ground    # Mahya bazlı bina toplam yüksekliği
        h_eaves = z_eaves - z_ground    # Saçak yüksekliği
        e = min(b, 2.0 * h_total)
        self.e= e
        return {
            "b": b,
            "d": d,
            "h": h_total,
            "h_eaves": h_eaves,
            "z0_ground": z_ground,
            "z1_eaves": z_eaves,
            "z2_ridge": z_ridge,
            "e": e
        }

    def _calculate_wind_parameters(self):
        self.q_b = 0.5 * self.rho * (self.v_b0 ** 2) / 1000.0
        self.z_ref = max(self.geometry["z2_ridge"], self.zmin)

        self.kr = 0.19 * ((self.z0 / 0.05) ** 0.07)
        self.cr = self.kr * math.log(self.z_ref / self.z0)
        self.Iv = self.kI / math.log(self.z_ref / self.z0)

        self.ce = (self.cr * self.ct) ** 2 * (1.0 + 7.0 * self.Iv)
        self.q_p = self.ce * self.q_b

    def get_summary(self) -> Dict[str, Any]:
        """Tüm özet sonuçları temiz bir dict olarak döndürür."""
        return {
            "Geometri": self.geometry,
            "Rüzgar Parametreleri": {
                "q_b (kN/m²)": round(self.q_b, 3),
                "q_p (kN/m²)": round(self.q_p, 3),
                "c_e": round(self.ce, 3),
                "I_v": round(self.Iv, 3),
                "c_r": round(self.cr, 3),
                "k_r": round(self.kr, 3),
                "z_ref (m)": round(self.z_ref, 3),
            },
            "Arazi": {
                "Kategori": self.terrain,
                "z0 (m)": self.z0,
                "zmin (m)": self.zmin
            }
        }

    def get_polygon_coords(self, polygon_points, points, scale_=1000.0):
        """Poligon noktalarını koordinatlara dönüştürür ve ölçekler."""
        coords = []
        for pt_name in polygon_points:
            if pt_name in points:
                pt = points[pt_name]
                coords.append([pt[0] / scale_, pt[1] / scale_, pt[2] / scale_])
            else:
                raise ValueError(f"Nokta '{pt_name}' bulunamadı!")
        return np.array(coords)

    def analysis_all_roof_wind(self, w):
        w = np.array(w, dtype=float)
        w_len = np.linalg.norm(w)
        if w_len < 1e-12:
            return
        w_norm = w / w_len
        self._analyze_surfaces(w)
        roof_surfaces = {}
        for surface_name, surface in self.surfaces_items.items():
            surface.analysis_(w, self)
            surface.global_leading = False
            surface.properties["global_leading"] = False
            if getattr(surface.surface_type, "value", surface.surface_type) == "ROOF":
                roof_surfaces[surface_name] = surface
                
        if not roof_surfaces:
            return

        # 1. Aşama: Bütün binalardaki exposed kenarların EN ÖN noktalarını kıyasla,
        # küresel rüzgarüstü sınır koordinatını (min_global_pos) bul.
        global_min_pos = float("inf")
        
        for surface in roof_surfaces.values():
            for edge in surface.edges.values():
                if not edge.exposed:
                    continue
                
                p1_pos = np.dot(np.array(edge.p1), w_norm)
                p2_pos = np.dot(np.array(edge.p2), w_norm)
                
                
                front_positions = sorted([p1_pos, p2_pos])  # min değer rüzgarda en öndedir
                
                if front_positions[0] < global_min_pos:
                    global_min_pos = front_positions[0]

        tol = 1e-4

        # 2. Aşama: Hem front_positions[0] hem de front_positions[1] aynı anda
        # en ön hizada olan kenarı barındıran yüzeyi global_leading yap.
        for surface_name, surface in roof_surfaces.items():
            is_leading = False
            
            for edge in surface.edges.values():
                if not edge.exposed:
                    continue

                p1_pos = np.dot(np.array(edge.p1), w_norm)
                p2_pos = np.dot(np.array(edge.p2), w_norm)
                
                # front_positions[0]: en ön nokta, front_positions[1]: en arka nokta
                front_positions = sorted([p1_pos, p2_pos])

                # ŞART: Kenarın SADECE ilk ucu değil, İKİNCİ UCU da (front_positions[1]) 
                # en ön hatta mı? (Yani kenar boydan boya ön cephede mi?)
                is_full_edge_at_front = (
                    abs(front_positions[0] - global_min_pos) <= tol and
                    abs(front_positions[1] - global_min_pos) <= tol
                )

                if is_full_edge_at_front:
                    # Nokta temaslarını elemek için kenar boyu kontrolü
                    edge_len = np.linalg.norm(np.array(edge.p2) - np.array(edge.p1))
                    if edge_len > 1e-2:
                        is_leading = True
                        break

            if is_leading:
                surface.global_leading = True
                surface.properties["global_leading"] = True
        return self.surfaces_items
                
    @staticmethod
    def calculate_obb_and_geometry(
        p1: List[float], 
        p2: List[float], 
        points_dict: Dict[str, List[float]]
    ) -> Optional[Dict[str, Any]]:
        """
        Seçilen P1-P2 doğrultusunu ve modeldeki tüm noktaları alarak
        rüzgara göre dönmüş sınırlayıcı kutu (OBB) ve B, d, H geometrisini türetir.
        """
        # 1. Rüzgara Paralel Doğrultu Vektörü (U) ve Dik Doğrultu Vektörü (V)
        du_x, du_y = p2[0] - p1[0], p2[1] - p1[1]
        L_u = math.sqrt(du_x**2 + du_y**2)
        if L_u < 1e-6:
            return None

        ux, uy = du_x / L_u, du_y / L_u  # Rüzgar yüzeyine paralel yön (U)
        vx, vy = -uy, ux                 # Rüzgarın esme yönü / Yapıya DİK yön (V)

        # 2. Tüm Bina Noktalarının U-V Eksenlerine ve Z Kotuna İzdüşümü
        u_vals, v_vals, z_vals = [], [], []

        for pt in points_dict.values():
            rx, ry, rz = pt[0] - p1[0], pt[1] - p1[1], pt[2]
            u_vals.append(rx * ux + ry * uy)
            v_vals.append(rx * vx + ry * vy)
            z_vals.append(rz)

        if not u_vals:
            return None

        u_min, u_max = min(u_vals), max(u_vals)
        v_min, v_max = min(v_vals), max(v_vals)
        z_min, z_max = min(z_vals), max(z_vals)

        # 3. Geometrik Mühendislik Parametreleri (TS EN 1991-1-4)
        B = u_max - u_min                # Rüzgar yönüne dik genişlik (b)
        d = v_max - v_min                # Rüzgar yönüne paralel derinlik (d)
        H = z_max - z_min                # Yapı yüksekliği (h)
        e = min(B, 2.0 * H)              # Kritik referans uzunluk e

        # 4. Z=0 Düzleminde Dönük Sınırlayıcı Kutu (OBB) Köşe Koordinatları
        c1 = [p1[0] + ux * u_min + vx * v_min, p1[1] + uy * u_min + vy * v_min, 0.0]
        c2 = [p1[0] + ux * u_max + vx * v_min, p1[1] + uy * u_max + vy * v_min, 0.0]
        c3 = [p1[0] + ux * u_max + vx * v_max, p1[1] + uy * u_max + vy * v_max, 0.0]
        c4 = [p1[0] + ux * u_min + vx * v_max, p1[1] + uy * u_min + vy * v_max, 0.0]

        # 5. Rüzgar Vektörünün Dışarıdan Kutunun Ön Yüzüne Saplanma Noktası (Z=0)
        u_mid = (u_min + u_max) / 2.0
        impact_point = [p1[0] + ux * u_mid + vx * v_min, p1[1] + uy * u_mid + vy * v_min, 0.0]

        return {
            "b": B,
            "d": d,
            "h": H,
            "e": e,
            "impact_point": impact_point,
            "wind_dir": [vx, vy, 0.0],
            "parallel_dir": [ux, uy, 0.0],
            "obb_corners": [c1, c2, c3, c4],
            "z0_ground": z_min,
            "z_max": z_max
        }
    @classmethod
    def generate_render_lines(cls, geom: Dict[str, Any]) -> List[List[List[float]]]:
        """
        View3D'nin (OpenGL) hiçbir hesap yapmadan doğrudan çizeceği
        saf [ [start_pt, end_pt], ... ] çizgi dizilerini üretir.
        """
        lines = []

        # A) Z=0 Düzleminde Dönük Sınırlayıcı Kutu (OBB - 4 Ana Kenar)
        c1, c2, c3, c4 = geom["obb_corners"]
        lines.extend([[c1, c2], [c2, c3], [c3, c4], [c4, c1]])

        # B) Rüzgar Oku (Bina Dışında, OBB Ön Yüzüne Saplanıyor)
        imp = geom["impact_point"]
        vx, vy = geom["wind_dir"][0], geom["wind_dir"][1]
        ux, uy = geom["parallel_dir"][0], geom["parallel_dir"][1]
        B = geom["b"]

        arrow_len = max(B * 0.35, 2.5)
        head_len, head_wing = arrow_len * 0.25, arrow_len * 0.15

        tail = [imp[0] - vx * arrow_len, imp[1] - vy * arrow_len, 0.0]
        h1 = [imp[0] - vx * head_len + ux * head_wing, imp[1] - vy * head_len + uy * head_wing, 0.0]
        h2 = [imp[0] - vx * head_len - ux * head_wing, imp[1] - vy * head_len - uy * head_wing, 0.0]

        lines.append([tail, imp])  # Gövde
        lines.append([imp, h1])    # Sol Ok Başı
        lines.append([imp, h2])    # Sağ Ok Başı

        # C) Rüzgar Okunun Arkasına 'W' Harfi Sembolü
        w_size = arrow_len * 0.12
        w_base = [tail[0] - vx * (w_size * 1.5), tail[1] - vy * (w_size * 1.5), 0.0]

        wp0 = [w_base[0] - ux * w_size, w_base[1] - uy * w_size, 0.0]
        wp1 = [w_base[0] - ux * (w_size * 0.5) - vx * w_size, w_base[1] - uy * (w_size * 0.5) - vy * w_size, 0.0]
        wp2 = [w_base[0], w_base[1], 0.0]
        wp3 = [w_base[0] + ux * (w_size * 0.5) - vx * w_size, w_base[1] + uy * (w_size * 0.5) - vy * w_size, 0.0]
        wp4 = [w_base[0] + ux * w_size, w_base[1] + uy * w_size, 0.0]

        lines.extend([[wp0, wp1], [wp1, wp2], [wp2, wp3], [wp3, wp4]])

        return lines
    

In [1]:
import numpy as np



def dict_tree(data, indent=""):
    lines = []

    items = list(data.items())

    for i, (key, value) in enumerate(items):
        last = i == len(items) - 1

        branch = "└── " if last else "├── "
        lines.append(f"{indent}{branch}{key}")

        if isinstance(value, dict):
            new_indent = indent + ("    " if last else "│   ")
            lines.append(dict_tree(value, new_indent))

        else:
            # Yukarıdaki key satırını value ile birleştir
            lines[-1] = f"{indent}{branch}{key} : {value}"

    return "\n".join(lines)


points = {
                "P1": (0.0, 0.0, 0.0),
                "P2": (12000.0, 0.0, 0.0),
                "P3": (12000.0, 8000.0, 0.0),
                "P4": (0.0, 8000.0, 0.0),
                "P5": (0.0, 0.0, 4000.0),
                "P6": (12000.0, 0.0, 4000.0),
                "P7": (12000.0, 8000.0, 4000.0),
                "P8": (0.0, 8000.0, 4000.0),
                "P9": (3000.0, 4000.0, 5000.0),
                "P10": (12000.0, 4000.0, 5000.0),
            }
    
polygons = {
    # "D1": ["P1", "P2", "P6", "P5"],
    # "D2": ["P1", "P5", "P8", "P4"],
    # "D3": ["P2", "P3", "P7", "P10", "P6"],
    # "D4": ["P3", "P4", "P8", "P7"],
    # "C1": ["P5", "P6", "P10", "P9"],
    "C2": ["P8", "P9", "P10", "P7"],
    # "C3": ["P8", "P5", "P9"],
}

building= BuildingWindEngine(points=points, polygons=polygons, scale_factor=1000)

    
            

print("\nbuilding analysis:\n")

wind_parameters= building.get_summary()
print("\nwind_parameters:\n", dict_tree(wind_parameters))

all_surfaces_w= {}
w_list= {"x+":[1,0,0], "y+":[0,1,0], "x-":[-1,0,0], "y-": [0,-1,0]}
for w in w_list:

    all_surfaces_w[w]= building.analysis_all_roof_wind(w_list[w])
    

for plane_name in polygons:

    
    print("\n>",plane_name)
    for w in w_list:
        print(f"\nWind vector: {w} : {w_list[w]}")
        plane1= all_surfaces_w[w][plane_name]
        for kk in ['surface_type', 'polygon', 'angle', 'pitch',  'wind_vector', 'polygon_direction_xy', 'wind_relation', 'global_leading', 'any_shared']:
            print(f" > {kk:20}: {plane1.properties.get(kk, '')}")
        if plane1.surface_type == SurfaceType.WALL:
            continue
            
        for ke_, ed_ in plane1.edges.items():
            print("\n > edge:",ke_)
            print("   > points:",ed_.p1, ed_.p2)
            print("   > vector_xy:",ed_.vector_xy, "| direction_xy:",ed_.direction_xy)
            print("   > angle:",ed_.angle, " | length:",ed_.length)
            print("   > vertical:",ed_.vertical)
            print("   > exposed:",ed_.exposed)
            print("   > leading:",ed_.leading)
            print("   > shared:",ed_.shared)
            print("   > same_axis:",ed_.same_axis)
            


ImportError: attempted relative import with no known parent package